<a href="https://colab.research.google.com/github/wajeeha-urooj/Pubmed-Evidence-Extractor/blob/main/PubMed_Evidence_Extractor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PubMed Evidence Extractor

This notebook takes a research question, searches PubMed, and turns the abstracts it finds into a table with one row per paper. For each paper the table records the study design, which population, exposure and outcome terms appear in the abstract, and the sentences that most likely report the main result. Every row keeps its PMID and DOI, so each entry can be checked against the original paper.

The example question comes from my thesis topic: does body mass index affect AMH and other hormone levels in women with polycystic ovary syndrome (PCOS). The search settings are in step 1, and the word lists in step 6 are the part to replace if you want to use the tool on a different topic.

The tool is rule based. It looks for words and patterns in abstracts and does not understand them. It cannot read full papers, and it cannot tell a positive result from a negative one. It is there to speed up screening and does not replace reading the papers.



## Setup

Biopython is the library that talks to PubMed, pandas holds the tables, and openpyxl lets pandas write Excel files.

In [ ]:
%pip install -q biopython pandas openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 35.7 MB/s eta 0:00:00


In [ ]:
import html
import json
import os
import re
import time
import unicodedata
from datetime import datetime, timezone
from pathlib import Path

import Bio
import pandas as pd
from Bio import Entrez

pd.set_option("display.max_colwidth", 200)
pd.set_option("display.width", 200)

## 1. Settings

Everything you might want to change is in the next cell. NCBI, the organisation that runs PubMed, asks every user for an email address so that they can get in touch if your requests cause trouble, and requests without one can be blocked. An API key is optional and only lets you send requests faster. Type your email into the cell, or set it as the NCBI_EMAIL environment variable, and clear it again before you push the notebook to GitHub.

The search uses a fixed range of publication years. PubMed keeps adding papers, so a search with no end date returns something different every time it is run, and a closed range makes the run repeatable. Results come back in PubMed's relevance order, so a small MAX_RESULTS keeps only the top of a longer list. The notebook prints how many papers matched in total so you can see how much was left out.

In [ ]:
QUERY = "PCOS AND BMI AND AMH"
START_YEAR = "2019"
END_YEAR = "2026"
MAX_RESULTS = 50

EXPORT_NAME = "pcos_bmi_amh_evidence"
OUTPUT_DIR = Path("results")

ENTREZ_EMAIL = "research@gmail.com"
NCBI_API_KEY = ""

ENTREZ_EMAIL = ENTREZ_EMAIL or os.environ.get("NCBI_EMAIL", "")
NCBI_API_KEY = NCBI_API_KEY or os.environ.get("NCBI_API_KEY", "")

if not ENTREZ_EMAIL or "example.com" in ENTREZ_EMAIL:
    raise ValueError("Put your real email address in ENTREZ_EMAIL (or set NCBI_EMAIL) before running.")

Entrez.email = ENTREZ_EMAIL
Entrez.tool = "pubmed-evidence-extractor"
if NCBI_API_KEY:
    Entrez.api_key = NCBI_API_KEY

# NCBI allows about 3 requests per second without an API key and about 10 with one.
REQUEST_PAUSE = 0.12 if NCBI_API_KEY else 0.34
BATCH_SIZE = 200

OUTPUT_DIR.mkdir(exist_ok=True)

print("Query:", QUERY)
print("Publication years:", START_YEAR, "to", END_YEAR)
print("Maximum papers:", MAX_RESULTS)

Query: PCOS AND BMI AND AMH
Publication years: 2019 to 2026
Maximum papers: 50


## 2. Search PubMed

A query like PCOS AND BMI AND AMH asks for papers that mention all three. PubMed answers with a list of PMIDs, which are just ID numbers, and the total number of matches. If a request fails for a temporary reason the function tries again up to three times before giving up.

In [ ]:
def _entrez_read(entrez_function, retries=3, **params):
    """Send one request to NCBI and parse the answer, retrying on temporary failures."""
    for attempt in range(1, retries + 1):
        try:
            handle = entrez_function(**params)
            try:
                return Entrez.read(handle)
            finally:
                handle.close()
        except (OSError, RuntimeError, ValueError) as error:
            if attempt == retries:
                raise
            print(f"Request failed ({error}). Trying again, attempt {attempt + 1} of {retries}.")
            time.sleep(2 * attempt)


def search_pubmed(query, retmax=50, mindate=None, maxdate=None, sort="relevance"):
    """Return the PMIDs matching a query, and the total number of matches in PubMed."""
    params = {"db": "pubmed", "term": query, "retmax": retmax, "sort": sort}
    if mindate or maxdate:
        params.update({
            "datetype": "pdat",
            "mindate": mindate or "1800",
            "maxdate": maxdate or "3000",
        })
    answer = _entrez_read(Entrez.esearch, **params)
    time.sleep(REQUEST_PAUSE)
    return list(answer.get("IdList", [])), int(answer.get("Count", 0))


pmids, total_matches = search_pubmed(QUERY, retmax=MAX_RESULTS, mindate=START_YEAR, maxdate=END_YEAR)

print("Papers matching in PubMed:", total_matches)
print("Papers retrieved:", len(pmids))
if total_matches > len(pmids):
    print(f"Only the top {len(pmids)} by relevance are used. Raise MAX_RESULTS to include more.")
if not pmids:
    raise ValueError("The search returned no papers. Check the query and the year range.")
print("First PMIDs:", pmids[:10])

Papers matching in PubMed: 248
Papers retrieved: 50
Only the top 50 by relevance are used. Raise MAX_RESULTS to include more.
First PMIDs: ['37528417', '38027110', '39378412', '39978319', '40806019', '40108611', '31414908', '42494866', '38753423', '40155948']


## 3. Download the records

A PMID is only a number. This step downloads the full record for each one: title, authors, journal, abstract and so on. The requests go out in batches of 200, with a short pause between them because NCBI limits how fast you can ask.

In [ ]:
def fetch_records(pmids):
    """Download the full PubMed record for each PMID."""
    records = []
    for start in range(0, len(pmids), BATCH_SIZE):
        batch = pmids[start:start + BATCH_SIZE]
        parsed = _entrez_read(Entrez.efetch, db="pubmed", id=",".join(batch), retmode="xml")
        records.extend(parsed.get("PubmedArticle", []) if hasattr(parsed, "get") else list(parsed))
        time.sleep(REQUEST_PAUSE)
    return records


records = fetch_records(pmids)
print("Records downloaded:", len(records), "of", len(pmids), "requested")

Records downloaded: 50 of 50 requested


## 4. Turn the records into a table

The downloaded records are nested XML, which is awkward to work with, so this step pulls out the fields the tool needs into a flat table with one row per paper.

Many abstracts are written in labelled sections such as Background, Methods, Results and Conclusions. I keep those sections apart, because a sentence from Results is much more likely to be a finding than a sentence from Background. Abstracts without labels are kept whole.

The clean_text function removes leftover formatting from the text. It only removes real HTML tags. A cruder version that deletes everything between a less-than sign and the next greater-than sign would cut out parts of sentences that contain both p < 0.05 and BMI > 25.

In [ ]:
def clean_text(text):
    """Remove leftover formatting from text and tidy the spacing."""
    if text is None:
        return ""
    if not isinstance(text, str) and pd.isna(text):
        return ""
    text = html.unescape(str(text))
    text = re.sub(r"</?[A-Za-z][^<>]*>", " ", text)
    text = text.replace("\u2009", " ").replace("\xa0", " ")
    return " ".join(text.split())


METHOD_WORDS = ("method", "design", "setting", "participant", "patient", "material", "subject")


def _section_kind(label, nlm_category=""):
    """Say whether an abstract section is methods, results or conclusions (or none of them)."""
    name = " ".join(re.sub(r"[^a-z]+", " ", label.lower()).split())
    if not name:
        return {"METHODS": "methods", "RESULTS": "results",
                "CONCLUSIONS": "conclusions"}.get(nlm_category.upper(), "")
    if any(word in name for word in METHOD_WORDS):
        return "methods"
    if "conclusion" in name or "interpretation" in name:
        return "conclusions"
    if "result" in name or "finding" in name:
        return "results"
    return ""


def _abstract_parts(article):
    """Return the whole abstract plus its Methods, Results and Conclusions text."""
    sections = {"methods": [], "results": [], "conclusions": []}
    whole = []
    for part in article.get("Abstract", {}).get("AbstractText", []):
        text = clean_text(part)
        if not text:
            continue
        attributes = getattr(part, "attributes", {}) or {}
        label = clean_text(attributes.get("Label", ""))
        kind = _section_kind(label, str(attributes.get("NlmCategory", "")))
        whole.append(f"{label}: {text}" if label else text)
        if kind:
            sections[kind].append(text)
    return {"abstract": " ".join(whole),
            "methods": " ".join(sections["methods"]),
            "results": " ".join(sections["results"]),
            "conclusions": " ".join(sections["conclusions"])}


def _doi(record, article):
    for article_id in record.get("PubmedData", {}).get("ArticleIdList", []):
        if getattr(article_id, "attributes", {}).get("IdType") == "doi":
            return str(article_id).strip()
    for location in article.get("ELocationID", []):
        if getattr(location, "attributes", {}).get("EIdType") == "doi":
            return str(location).strip()
    return ""


def _authors(article):
    names = []
    for author in article.get("AuthorList", []):
        last_name = author.get("LastName")
        if last_name:
            names.append(f"{last_name} {author.get('Initials', '')}".strip())
        elif author.get("CollectiveName"):
            names.append(clean_text(author["CollectiveName"]))
    return "; ".join(names)


def _year(article):
    pubdate = article.get("Journal", {}).get("JournalIssue", {}).get("PubDate", {})
    if "Year" in pubdate:
        return str(pubdate["Year"])
    match = re.search(r"\b(?:19|20)\d{2}\b", str(pubdate.get("MedlineDate", "")))
    if match:
        return match.group(0)
    for date in article.get("ArticleDate", []):
        if "Year" in date:
            return str(date["Year"])
    return ""


COLUMNS = ["PMID", "DOI", "Title", "Authors", "Year", "Journal", "Publication_Types",
           "Abstract", "Methods", "Results", "Conclusions"]


def parse_records(records):
    """Turn raw PubMed records into a plain table, one row per paper."""
    rows = []
    for record in records:
        citation = record.get("MedlineCitation", {})
        article = citation.get("Article", {})
        parts = _abstract_parts(article)
        rows.append({
            "PMID": str(citation.get("PMID", "")),
            "DOI": _doi(record, article),
            "Title": clean_text(article.get("ArticleTitle", "")),
            "Authors": _authors(article),
            "Year": _year(article),
            "Journal": clean_text(article.get("Journal", {}).get("Title", "")),
            "Publication_Types": "; ".join(str(p) for p in article.get("PublicationTypeList", [])),
            "Abstract": parts["abstract"],
            "Methods": parts["methods"],
            "Results": parts["results"],
            "Conclusions": parts["conclusions"],
        })
    return pd.DataFrame(rows, columns=COLUMNS)


papers = parse_records(records)
print("Rows:", len(papers))
papers[["PMID", "DOI", "Title", "Year", "Journal", "Publication_Types"]].head(5)

Rows: 50


,PMID,DOI,Title,Year,Journal,Publication_Types
0,37528417,10.1186/s12958-023-01120-7,Women with PCOS who undergo IVF: a comprehensive review of therapeutic strategies for successful outcomes.,2023,Reproductive biology and endocrinology : RB&E,Journal Article; Review
1,38027110,10.3389/fendo.2023.1183060,"Polycystic ovary syndrome and recurrent pregnancy loss, a review of literature.",2023,Frontiers in endocrinology,Meta-Analysis; Systematic Review; Journal Article
2,39378412,10.1093/humupd/dmae030,Functional hypothalamic amenorrhoea and polycystic ovarian morphology: a narrative review about an intriguing association.,2025,Human reproduction update,Journal Article; Review
3,39978319,10.1159/000543941,"Impact of Ketogenic Diet on Weight, Metabolic, and Endocrine Parameters in Women with Polycystic Ovary Syndrome: A Systematic Review and Meta-Analysis.",2025,Gynecologic and obstetric investigation,Journal Article; Systematic Review; Meta-Analysis
4,40806019,10.3390/nu17152436,"Effect of Intermittent Fasting on Anthropometric Measurements, Metabolic Profile, and Hormones in Women with Polycystic Ovary Syndrome: A Systematic Review and Meta-Analysis.",2025,Nutrients,Journal Article; Systematic Review; Meta-Analysis


A paper without an abstract has nothing for the tool to read, so it is worth knowing how many there are before going further.

In [ ]:
def count_blank(table, column):
    return int(table[column].fillna("").astype(str).str.strip().eq("").sum())


print("Papers with no abstract:", count_blank(papers, "Abstract"), "of", len(papers))
print("Papers with no DOI:", count_blank(papers, "DOI"), "of", len(papers))
print("Papers with labelled Results or Conclusions sections:",
      int((papers["Results"].str.strip().ne("") | papers["Conclusions"].str.strip().ne("")).sum()), "of", len(papers))
print("Publication years range from", papers["Year"].replace("", pd.NA).dropna().min(),
      "to", papers["Year"].replace("", pd.NA).dropna().max())

Papers with no abstract: 0 of 50
Papers with no DOI: 0 of 50
Papers with labelled Results or Conclusions sections: 39 of 50
Publication years range from 2019 to 2026


## 5. Read one abstract first

Rules for text should be written after looking at real text. This prints one abstract, followed by the sections the parser found in it.

In [ ]:
with_abstract = papers[papers["Abstract"].str.strip() != ""]

if with_abstract.empty:
    print("None of the retrieved papers has an abstract.")
else:
    example = with_abstract.iloc[0]
    print("TITLE")
    print(example["Title"])
    print("\nABSTRACT")
    print(example["Abstract"][:2500])
    for section in ("Methods", "Results", "Conclusions"):
        print(f"\n{section.upper()} (as found by the parser)")
        print(example[section] or "[none found]")

TITLE
Women with PCOS who undergo IVF: a comprehensive review of therapeutic strategies for successful outcomes.

ABSTRACT
Polycystic ovarian syndrome (PCOS) is a widespread syndrome that poses unique challenges and constraints to the field of assisted reproductive technology. This condition is the most common cause of anovulation among infertile couples. Debate exists over the best therapeutic course of action when patients with PCOS proceed to IVF. In this review, we evaluate the best-performing and safest methods of IVF preparation, ovarian stimulation, trigger method for maturation of stimulated egg growth, and planning for embryo transfer. Pre-IVF considerations include being aware of individual AMH and vitamin D levels as well as BMI prior to selecting an ovarian stimulation protocol. Numerous supplements such as myo-inositol complement the benefits of lifestyle change and may enhance IVF performance including oocyte yield and pregnancy rate. Concerning stimulation protocols, ant

## 6. Word lists

The tool finds terms by matching words from lists. There are three kinds. Population is who was studied. Exposure is the thing whose effect is being tested, such as BMI or a diet. Outcome is what was measured, such as a hormone level or a pregnancy rate.

Each term has a name and the spellings that count as that term. Body mass index and BMI end up as one entry, BMI, and Müllerian written with or without the umlaut both count as AMH. A plural is matched automatically. Matching is on whole words, so LH is never found inside a longer word. When one spelling contains another, the longer one is claimed first. That is why follicle-stimulating hormone is counted as FSH and not also as follicle.

A term being found means the word appears in the abstract. It does not mean the study was about it. An abstract that mentions BMI once in passing still gets BMI in its Exposure_Terms, which is why the columns are called Terms. The validation notebook measures how big that gap is. The same term can sit in both the exposure and outcome lists (insulin resistance does), because the tool cannot tell which role it played in a given paper.

These lists are written for the PCOS example. For another topic they are the part to replace.

In [ ]:
POPULATION_TERMS = {
    "women": ["women", "woman"],
    "men": ["men"],
    "patients": ["patient"],
    "participants": ["participant"],
    "adolescents": ["adolescent"],
    "adults": ["adult"],
    "children": ["children", "child"],
    "subjects": ["subjects"],
    "controls": ["controls"],
    "infertile": ["infertile"],
    "premenopausal": ["premenopausal", "pre-menopausal"],
    "postmenopausal": ["postmenopausal", "post-menopausal"],
}

EXPOSURE_TERMS = {
    "BMI": ["BMI", "body mass index"],
    "obesity": ["obesity", "obese"],
    "overweight": ["overweight"],
    "adiposity": ["adiposity"],
    "waist circumference": ["waist circumference"],
    "weight loss": ["weight loss"],
    "weight gain": ["weight gain"],
    "physical activity": ["physical activity"],
    "exercise": ["exercise"],
    "diet": ["diet", "dietary"],
    "metformin": ["metformin"],
    "insulin resistance": ["insulin resistance"],
    "hyperinsulinemia": ["hyperinsulinemia", "hyperinsulinaemia"],
    "hyperandrogenism": ["hyperandrogenism", "hyperandrogenemia", "hyperandrogenaemia"],
    "testosterone": ["testosterone"],
    "SHBG": ["SHBG", "sex hormone-binding globulin", "sex hormone binding globulin"],
    "smoking": ["smoking"],
    "intermittent fasting": ["intermittent fasting"],
    "ketogenic diet": ["ketogenic diet"],
    "low-carbohydrate diet": ["low-carbohydrate diet", "low carbohydrate diet"],
    "Mediterranean diet": ["Mediterranean diet"],
}

OUTCOME_TERMS = {
    "AMH": ["AMH", "anti-Mullerian hormone", "antimullerian hormone", "anti Mullerian hormone"],
    "LH": ["LH", "luteinizing hormone", "luteinising hormone"],
    "FSH": ["FSH", "follicle-stimulating hormone", "follicle stimulating hormone"],
    "androgen": ["androgen"],
    "estradiol": ["estradiol", "oestradiol"],
    "insulin resistance": ["insulin resistance"],
    "HOMA-IR": ["HOMA-IR", "HOMA IR"],
    "ovarian volume": ["ovarian volume"],
    "antral follicle count": ["antral follicle count", "AFC"],
    "follicle": ["follicle"],
    "ovulation": ["ovulation", "ovulatory"],
    "menstrual": ["menstrual"],
    "fertility": ["fertility"],
    "pregnancy": ["pregnancy", "pregnancies"],
    "live birth": ["live birth"],
    "miscarriage": ["miscarriage"],
    "oocyte": ["oocyte"],
    "endometrial": ["endometrial"],
}

## 7. Study design list

A study design is the shape of a study: whether it compared two groups, followed people over time, pooled the results of other studies, and so on.

Three places are checked for a design. PubMed's own Publication Type labels come first, then the title, then the abstract. Meta-analysis, systematic review, clinical trial and review are only accepted from the labels and the title. Otherwise an ordinary study that mentions a meta-analysis in its background would be labelled as one. In the abstract the Methods section is used when there is one, and the whole abstract when there is not.

If more than one design is found, the one higher in the list below wins, so a meta-analysis beats a cohort study. Diagnostic accuracy sits near the bottom because ROC analysis is a method that studies of many designs use, and an explicit statement such as cross-sectional study should win over it.

In [ ]:
# label, spellings, and whether the abstract text may be used (False = only PubMed's labels and the title)
STUDY_DESIGNS = [
    ("Meta-analysis", ["meta-analysis", "meta-analyses", "meta analysis", "metaanalysis"], False),
    ("Systematic review", ["systematic review"], False),
    ("Randomised controlled trial", ["randomized controlled trial", "randomised controlled trial",
                                     "randomized clinical trial", "randomised clinical trial",
                                     "randomized trial", "randomised trial",
                                     "randomly assigned", "randomly allocated"], True),
    ("Clinical trial (not randomised or not stated)", ["clinical trial"], False),
    ("Prospective cohort study", ["prospective cohort"], True),
    ("Retrospective cohort study", ["retrospective cohort"], True),
    ("Cohort study", ["cohort study", "cohort studies"], True),
    ("Case-control study", ["case-control", "case control"], True),
    ("Cross-sectional study", ["cross-sectional", "cross sectional"], True),
    ("Longitudinal study", ["longitudinal study", "longitudinal studies"], True),
    ("Prospective study", ["prospective study", "prospective studies"], True),
    ("Retrospective study", ["retrospective study", "retrospective studies"], True),
    ("Observational study", ["observational study", "observational studies"], True),
    ("Diagnostic accuracy study", ["diagnostic accuracy", "receiver operating characteristic",
                                   "sensitivity and specificity", "roc curve"], True),
    ("Review (not systematic)", ["review"], False),
]

## 8. Matching terms and designs

For matching, text is lower-cased and accents and different kinds of dash are simplified, so Müllerian and Mullerian become the same word. The find_terms function returns every term whose spelling appears as a whole word. The detect_study_design function follows the rules from step 7.

In [ ]:
def normalise_for_matching(text):
    """Lower-case, drop accents and turn the different dash characters into a plain hyphen."""
    text = unicodedata.normalize("NFKD", clean_text(text))
    text = "".join(character for character in text if not unicodedata.combining(character))
    for dash in ("\u2010", "\u2011", "\u2012", "\u2013", "\u2014", "\u2212"):
        text = text.replace(dash, "-")
    return text.lower()


def phrase_pattern(spellings):
    """A pattern that matches any of the spellings as whole words, with an optional plural s."""
    alternatives = "|".join(re.escape(normalise_for_matching(s)) for s in spellings)
    return re.compile(rf"(?<![a-z0-9])(?:{alternatives})s?(?![a-z0-9])")


def find_terms(text, term_dict):
    """Return the names of the terms whose spellings appear in the text, joined by semicolons."""
    remaining = normalise_for_matching(text)
    if not remaining:
        return ""
    spellings = sorted(((spelling, name) for name, options in term_dict.items() for spelling in options),
                       key=lambda item: -len(item[0]))
    found = set()
    for spelling, name in spellings:
        pattern = phrase_pattern([spelling])
        if pattern.search(remaining):
            found.add(name)
            remaining = pattern.sub(" ", remaining)
    return "; ".join(name for name in term_dict if name in found)


def detect_study_design(publication_types, title, abstract_text):
    """Return the highest ranked study design found, or an empty string."""
    sources = [(publication_types, True), (title, True), (abstract_text, False)]
    best = None
    for text, trusted in sources:
        normalised = normalise_for_matching(text)
        if not normalised:
            continue
        for rank, (label, spellings, allowed_in_abstract) in enumerate(STUDY_DESIGNS):
            if not trusted and not allowed_in_abstract:
                continue
            if phrase_pattern(spellings).search(normalised):
                if best is None or rank < best[0]:
                    best = (rank, label)
                break
    return best[1] if best else ""

## 9. Finding the main result

An abstract mixes background, methods and results. To pick out the sentences that read like a reported result, every sentence gets a score. Statistical language raises it: a p value, a confidence interval, an effect size such as an odds ratio, and words like significant, associated or higher. Background language lowers it: we aimed to, little is known, further research is needed.

When the abstract has Results or Conclusions sections, only those are scored. Otherwise the whole abstract is scored. The two best sentences are returned in their original order, copied word for word from the abstract, and the Finding_Basis column says where they came from.

This finds sentences that read like results. It does not judge whether the result was positive or negative, and sometimes it will pick the wrong sentence.

In [ ]:
FINDING_CUES = [re.compile(r"(?<![a-z])" + re.escape(stem)) for stem in (
    "significan", "associat", "correlat", "increase", "decrease", "higher", "lower",
    "greater", "reduc", "predict", "did not differ", "no difference")]

REPORTING_VERBS = re.compile(r"\b(?:found|showed|demonstrated|revealed|observed|reported)\b")

BACKGROUND_CUES = (
    "is a common", "is a prevalent", "is characterised by", "is characterized by",
    "remains poorly understood", "remains unclear", "little is known",
    "we aimed", "this study aimed", "aimed to", "the aim of this", "the aim was",
    "the objective of this", "the objective was", "the purpose of this", "the purpose was",
    "we investigated", "we assessed", "we evaluated", "we examined",
    "were recruited", "were enrolled", "data were collected",
    "further studies", "further research", "future research", "more research")

P_VALUE = re.compile(r"\bp\s*[<>=\u2264\u2265]\s*0?\.\d+")

# Phrases are matched in any case. The short abbreviations must be capitals, or the word "or" would match OR.
EFFECT_SIZE = re.compile(
    r"\b(?i:odds ratio|hazard ratio|risk ratio|relative risk|mean difference|"
    r"beta coefficient|regression coefficient)\b"
    r"|\b(?:OR|HR|RR|AUC)\b"
    r"|\b(?:r|rs|rho)\s*=\s*[-\u2212]?\d")

CONFIDENCE_INTERVAL = re.compile(r"\b95\s*%\s*(?:ci|confidence interval)\b|\bconfidence interval\b")

SENTENCE_SPLIT = re.compile(r"(?<=[.!?])\s+(?=[A-Z0-9])")

LABEL_AT_START = re.compile(
    r"^(?:background|objectives?|aims?|purpose|introduction|methods?|materials and methods|design|setting|"
    r"patients|participants|results?|conclusions?|findings?)\s*:\s*", re.IGNORECASE)


def score_sentence(sentence):
    """Score how much a sentence reads like a reported result."""
    lowered = sentence.lower()
    score = 2 * sum(1 for cue in FINDING_CUES if cue.search(lowered))
    score += len(set(REPORTING_VERBS.findall(lowered)))
    if P_VALUE.search(lowered):
        score += 5
    if EFFECT_SIZE.search(sentence):
        score += 4
    if CONFIDENCE_INTERVAL.search(lowered):
        score += 4
    score -= 6 * sum(1 for cue in BACKGROUND_CUES if cue in lowered)
    return score


def extract_main_finding(parts, max_sentences=2):
    """Return the best scoring sentences from a list of texts, in the order they were written."""
    scored = []
    position = 0
    for part in parts:
        for raw in SENTENCE_SPLIT.split(clean_text(part)):
            sentence = LABEL_AT_START.sub("", raw.strip()).strip()
            position += 1
            if len(sentence.split()) < 5:
                continue
            score = score_sentence(sentence)
            if score > 0:
                scored.append((score, position, sentence))
    best = sorted(scored, key=lambda item: (-item[0], item[1]))[:max_sentences]
    return [sentence for _, _, sentence in sorted(best, key=lambda item: item[1])]


def pick_main_finding(row, max_sentences=2):
    """Try the Results and Conclusions sections first and the whole abstract second."""
    structured = [text for text in (row["Results"], row["Conclusions"]) if str(text).strip()]
    if structured:
        sentences = extract_main_finding(structured, max_sentences)
        if sentences:
            if len(structured) == 2:
                return sentences, "Results and Conclusions"
            return sentences, "Results only" if str(row["Results"]).strip() else "Conclusions only"
    if str(row["Abstract"]).strip():
        sentences = extract_main_finding([row["Abstract"]], max_sentences)
        if sentences:
            return sentences, "Whole abstract"
        return [], "No finding detected"
    return [], "No abstract"

## 10. Build the evidence table

This runs everything above on every paper at once. The Evidence_Source column joins the PMID and the DOI, so every row can be traced back to its paper.

In [ ]:
OUTPUT_COLUMNS = ["PMID", "DOI", "Title", "Authors", "Year", "Journal", "Study_Design",
                  "Population_Terms", "Exposure_Terms", "Outcome_Terms",
                  "Main_Finding", "Finding_Basis", "Evidence_Source"]


def extract_evidence(papers, population_terms=POPULATION_TERMS, exposure_terms=EXPOSURE_TERMS,
                     outcome_terms=OUTCOME_TERMS, max_finding_sentences=2):
    """Add design, term and finding columns to the table of papers."""
    evidence = papers.copy()
    evidence["Clean_Abstract"] = evidence["Abstract"].apply(clean_text)

    has_methods = evidence["Methods"].str.strip() != ""
    design_text = evidence["Methods"].where(has_methods, evidence["Clean_Abstract"])
    evidence["Study_Design"] = [
        detect_study_design(types, title, text)
        for types, title, text in zip(evidence["Publication_Types"], evidence["Title"], design_text)]

    evidence["Population_Terms"] = evidence["Clean_Abstract"].apply(lambda t: find_terms(t, population_terms))
    evidence["Exposure_Terms"] = evidence["Clean_Abstract"].apply(lambda t: find_terms(t, exposure_terms))
    evidence["Outcome_Terms"] = evidence["Clean_Abstract"].apply(lambda t: find_terms(t, outcome_terms))

    findings = [pick_main_finding(row, max_finding_sentences) for row in evidence.to_dict("records")]
    evidence["Finding_Sentences"] = [sentences for sentences, _ in findings]
    evidence["Main_Finding"] = [" ".join(sentences) for sentences, _ in findings]
    evidence["Finding_Basis"] = [basis for _, basis in findings]

    evidence["Evidence_Source"] = [
        f"PMID: {pmid}" + (f" | DOI: {doi}" if str(doi).strip() else "")
        for pmid, doi in zip(evidence["PMID"], evidence["DOI"])]
    return evidence

In [ ]:
evidence = extract_evidence(papers)

evidence[["PMID", "Study_Design", "Population_Terms", "Exposure_Terms", "Outcome_Terms",
          "Main_Finding", "Finding_Basis"]].head(10)

,PMID,Study_Design,Population_Terms,Exposure_Terms,Outcome_Terms,Main_Finding,Finding_Basis
0,37528417,Review (not systematic),women; patients; infertile,BMI; metformin,AMH; pregnancy; oocyte,"Following ovarian stimulation, PCOS patients typically undergo programmed frozen embryo transfer (FET) cycles which are more conducive for women with irregular cycles, but likely carry a higher ri...",Whole abstract
1,38027110,Meta-analysis,patients,BMI; obesity; insulin resistance; hyperandrogenism; SHBG,AMH; insulin resistance; pregnancy; live birth; miscarriage,PCOS is a syndrome of ovarian dysfunction associated with recurrent pregnancy loss. Several correlating factors have been investigated that influence the risk of pregnancy loss in PCOS.,Whole abstract
2,39378412,Review (not systematic),women; patients; controls,BMI; exercise; hyperandrogenism; testosterone; SHBG,AMH; LH; follicle,"OUTCOMES: The prevalence of PCOM in women with FHA varied from 41.9% to 46.7%, which is higher than in healthy non-PCOS controls. While oestrogen deficiency is common to both groups of patients, F...",Whole abstract
3,39978319,Meta-analysis,women; patients,BMI; obesity; overweight; weight loss; diet; insulin resistance; hyperinsulinemia; testosterone; SHBG; ketogenic diet; low-carbohydrate diet,AMH; LH; FSH; androgen; insulin resistance; HOMA-IR; ovulation; menstrual; pregnancy; live birth,"The results of the meta-analysis showed that after KD the patients had a significant weight loss (standard mean difference or SMD 1.31 kg [95% CI: 0.45, 2.17] p = 0.003) and lower BMI (SMD 1.27 kg...",Results and Conclusions
4,40806019,Meta-analysis,women,BMI; diet; insulin resistance; hyperandrogenism; testosterone; SHBG; intermittent fasting,AMH; androgen; insulin resistance; HOMA-IR,"IF significantly reduced body weight (MD = -4.25 kg, 95% CI: -7.71, -0.79; p = 0.02), BMI (MD = -2.05 kg/m 2 , 95% CI: -3.26, -0.85; p = 0.0008), fasting blood glucose (FBG; MD = -2.86 mg/dL, 95% ...",Results and Conclusions
5,40108611,Cross-sectional study,women; adults,BMI; obesity; overweight; adiposity; insulin resistance; testosterone; SHBG,AMH; androgen; insulin resistance; menstrual,AMH was negatively associated with age. AMH was not associated with BMI or insulin resistance.,Results and Conclusions
6,31414908,Meta-analysis,,BMI; hyperandrogenism,AMH,"The pooled sensitivity, specificity, and diagnostic odds ratio (DOR) for AMH alone detecting PCOS were 0.76 (95% confidence interval [CI] 0.71 to 0.81), 0.86 (95% CI 0.82 to 0.90) and 20 (95% CI 1...",Whole abstract
7,42494866,Retrospective study,women; controls,BMI; insulin resistance,AMH; insulin resistance; HOMA-IR,A negative association between AMH concentration and both fasting insulin and HOMA-IR was revealed. Subsequent analysis further demonstrated that the relationship between HOMA-IR and AMH is indepe...,Results and Conclusions
8,38753423,,women; participants,BMI; hyperandrogenism; testosterone; SHBG,AMH; LH; FSH; estradiol; follicle; ovulation,"In addition to high LH and SHBG levels, the reproductive subtype had the highest TFC and levels of AMH (all P < .001). In addition to high BMI and insulin levels, the metabolic subtype had higher ...",Results and Conclusions
9,40155948,,women; patients,BMI; testosterone,AMH; LH; androgen; antral follicle count; follicle; pregnancy; miscarriage,"In the PCOS group, hormonal and metabolic parameters such as insulin, blood lipids, luteinizing hormone (LH), anti-Müllerian hormone (AMH) and antral follicle counting (AFC) were significantly hig...",Results and Conclusions


### Excluding papers that do not fit

Some papers a search returns do not fit the question, for example mouse or cell studies when the review is about patients. Add their PMIDs to the list below after reading the titles. Excluded papers are removed before anything is counted or saved. The list is empty in the example.

In [ ]:
EXCLUDE_PMIDS = []


def apply_exclusions(table, exclude_pmids):
    """Drop the papers whose PMID is on the exclusion list."""
    excluded = {str(pmid).strip() for pmid in exclude_pmids}
    return table[~table["PMID"].astype(str).isin(excluded)].reset_index(drop=True)


evidence = apply_exclusions(evidence, EXCLUDE_PMIDS)
results = evidence[OUTPUT_COLUMNS]

print(len(results), "papers in the evidence table")

50 papers in the evidence table


## 11. How complete is the table

Coverage is the share of papers where a field has a value. It says nothing about whether the value is right, only that something was found. A field can be 100 percent filled and still be wrong, and the validation notebook is where correctness gets checked.

When a field is empty there are two possible reasons. Either the abstract really does not say, which is a correct result, or the word lists do not cover what the abstract said, which is a gap worth fixing. The only way to tell them apart is to read a few of the empty ones.

In [ ]:
EVIDENCE_FIELDS = ["Study_Design", "Population_Terms", "Exposure_Terms", "Outcome_Terms", "Main_Finding", "DOI"]


def field_coverage(table, fields=EVIDENCE_FIELDS):
    """Share of papers with a non-empty value in each field."""
    total = len(table)
    rows = []
    for field in fields:
        if field not in table.columns:
            continue
        filled = int(table[field].fillna("").astype(str).str.strip().ne("").sum())
        rows.append({"Field": field, "Filled": filled, "Total": total,
                     "Coverage_%": round(100 * filled / total, 1) if total else 0.0})
    return pd.DataFrame(rows)


field_coverage(results)

,Field,Filled,Total,Coverage_%
0,Study_Design,34,50,68.0
1,Population_Terms,49,50,98.0
2,Exposure_Terms,50,50,100.0
3,Outcome_Terms,50,50,100.0
4,Main_Finding,50,50,100.0
5,DOI,50,50,100.0


In [ ]:
no_design = results[results["Study_Design"].str.strip() == ""]
print("Papers with no design detected:", len(no_design))
no_design[["PMID", "Title"]].head(20)

Papers with no design detected: 16


,PMID,Title
8,38753423,Clustering Identifies Subtypes With Different Phenotypic Characteristics in Women With Polycystic Ovary Syndrome.
9,40155948,Hormonal and metabolic influences on outcomes in PCOS undergoing assisted reproduction: the role of BMI in fresh embryo transfers.
10,38409361,BOP1 contributes to the activation of autophagy in polycystic ovary syndrome via nucleolar stress response.
11,40781325,Mechanistic role of the KRTAP5-AS1/miR-199b-5p/CYP19A1 axis in polycystic ovary syndrome pathogenesis.
13,38506476,Serum 25-hydroxyvitamin D is associated with homocysteine in infertile patients with polycystic ovary syndrome (PCOS).
14,40624700,Polycystic ovary syndrome and excessive body weight impact independently and synergically on fertility treatment outcomes.
15,34427289,"Effects of serum irisin, neuregulin 4, and weight management on obese adolescent girls with polycystic ovary syndrome."
16,37020210,AMH predicts miscarriage in non-PCOS but not in PCOS related infertility ART cycles.
17,41514462,Acupuncture of polycystic ovary syndrome: delving into bile acid metabolism.
35,35449841,Higher Chronic Endometritis Incidences within Infertile Polycystic Ovary Syndrome Clinical Cases.


## 12. The same steps as one function

Steps 2 to 10 are wrapped into a single function, so a new question needs one call instead of running each step by hand.

In [ ]:
def run_pipeline(query, max_results=50, mindate=None, maxdate=None,
                 population_terms=POPULATION_TERMS, exposure_terms=EXPOSURE_TERMS,
                 outcome_terms=OUTCOME_TERMS, exclude_pmids=()):
    """Search PubMed and return a finished evidence table."""
    found_pmids, _ = search_pubmed(query, retmax=max_results, mindate=mindate, maxdate=maxdate)
    if not found_pmids:
        return pd.DataFrame(columns=OUTPUT_COLUMNS)
    found_papers = parse_records(fetch_records(found_pmids))
    if found_papers.empty:
        return pd.DataFrame(columns=OUTPUT_COLUMNS)
    table = extract_evidence(found_papers, population_terms, exposure_terms, outcome_terms)
    return apply_exclusions(table, exclude_pmids)[OUTPUT_COLUMNS]

### Trying other questions

The pipeline works on any PubMed query, but the word lists do not carry over, because they were written for PCOS. The hypertension example passes its own exposure and outcome lists. The other two searches use the PCOS lists on purpose, so the coverage numbers show what happens when the lists do not fit the topic.

In [ ]:
HYPERTENSION_EXPOSURES = {
    "sodium": ["sodium"],
    "salt intake": ["salt intake", "dietary salt"],
    "DASH diet": ["DASH diet"],
}
HYPERTENSION_OUTCOMES = {
    "systolic blood pressure": ["systolic blood pressure", "SBP"],
    "diastolic blood pressure": ["diastolic blood pressure", "DBP"],
    "stroke": ["stroke"],
    "cardiovascular events": ["cardiovascular events", "cardiovascular disease"],
}

test_cases = [
    ("PCOS AND insulin resistance", {}),
    ("endometriosis AND infertility", {}),
    ("hypertension AND sodium intake",
     {"exposure_terms": HYPERTENSION_EXPOSURES, "outcome_terms": HYPERTENSION_OUTCOMES}),
]

summary = []
for test_query, extra_settings in test_cases:
    table = run_pipeline(test_query, max_results=20, **extra_settings)
    coverage = field_coverage(table).set_index("Field")["Coverage_%"]
    summary.append({
        "Query": test_query,
        "Papers": len(table),
        "Study_Design_%": coverage.get("Study_Design", 0),
        "Exposure_Terms_%": coverage.get("Exposure_Terms", 0),
        "Outcome_Terms_%": coverage.get("Outcome_Terms", 0),
        "Main_Finding_%": coverage.get("Main_Finding", 0),
    })

pd.DataFrame(summary)

,Query,Papers,Study_Design_%,Exposure_Terms_%,Outcome_Terms_%,Main_Finding_%
0,PCOS AND insulin resistance,20,100.0,100.0,100.0,90.0
1,endometriosis AND infertility,20,60.0,5.0,55.0,75.0
2,hypertension AND sodium intake,20,85.0,100.0,30.0,100.0


## 13. Save the results

The table is saved as CSV, Excel and JSON in the results folder. A small run_info.json file records the query, the year range, how many papers PubMed matched, when the search was run and the package versions. PubMed keeps changing, so the same query can return more papers later, and this file says what the table was built from. The word lists are saved too, so the validation notebook can check hand annotations against them.

In [ ]:
CSV_PATH = OUTPUT_DIR / f"{EXPORT_NAME}.csv"
EXCEL_PATH = OUTPUT_DIR / f"{EXPORT_NAME}.xlsx"
JSON_PATH = OUTPUT_DIR / f"{EXPORT_NAME}.json"
RUN_INFO_PATH = OUTPUT_DIR / "run_info.json"
TERM_LISTS_PATH = OUTPUT_DIR / "term_lists.json"

results.to_csv(CSV_PATH, index=False)
results.to_excel(EXCEL_PATH, index=False)
JSON_PATH.write_text(json.dumps(results.to_dict("records"), indent=2, ensure_ascii=False), encoding="utf-8")

run_info = {
    "query": QUERY,
    "publication_years": [START_YEAR, END_YEAR],
    "sorted_by": "relevance",
    "max_results": MAX_RESULTS,
    "papers_matching_in_pubmed": total_matches,
    "records_downloaded": len(records),
    "papers_in_table": len(results),
    "excluded_pmids": [str(p) for p in EXCLUDE_PMIDS],
    "search_run_utc": datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M"),
    "biopython_version": Bio.__version__,
    "pandas_version": pd.__version__,
}
RUN_INFO_PATH.write_text(json.dumps(run_info, indent=2), encoding="utf-8")

term_lists = {
    "Study_Design": [label for label, _, _ in STUDY_DESIGNS],
    "Population_Terms": POPULATION_TERMS,
    "Exposure_Terms": EXPOSURE_TERMS,
    "Outcome_Terms": OUTCOME_TERMS,
}
TERM_LISTS_PATH.write_text(json.dumps(term_lists, indent=2, ensure_ascii=False), encoding="utf-8")

print("Files written to", OUTPUT_DIR)
for path in (CSV_PATH, EXCEL_PATH, JSON_PATH, RUN_INFO_PATH, TERM_LISTS_PATH):
    print(" ", path.name)

Files written to results
  pcos_bmi_amh_evidence.csv
  pcos_bmi_amh_evidence.xlsx
  pcos_bmi_amh_evidence.json
  run_info.json
  term_lists.json


## 14. Final checks

These checks confirm that the table is complete and traceable. The required columns exist, every row has a PMID and a source string, no PMID appears twice, every extracted finding can be found word for word in its own abstract, and the saved files can be read back. They do not show that the extraction is correct. That is what the manual validation is for.

In [ ]:
def findings_are_verbatim(row):
    return all(sentence in row["Clean_Abstract"] for sentence in row["Finding_Sentences"])


saved_csv = pd.read_csv(CSV_PATH, dtype=str, keep_default_na=False)
saved_json = json.loads(JSON_PATH.read_text(encoding="utf-8"))

checks = {
    "Records were downloaded": len(records) > 0,
    "Evidence table has rows": len(results) > 0,
    "All required columns are present": all(column in results.columns for column in OUTPUT_COLUMNS),
    "Every row has a PMID": results["PMID"].astype(str).str.strip().ne("").all(),
    "No PMID appears twice": results["PMID"].is_unique,
    "Every row has an evidence source": results["Evidence_Source"].astype(str).str.strip().ne("").all(),
    "Every finding appears word for word in its abstract": all(findings_are_verbatim(r) for r in evidence.to_dict("records")),
    "CSV can be read back with the same number of rows": len(saved_csv) == len(results),
    "JSON can be read back with the same number of rows": len(saved_json) == len(results),
    "Excel file exists": EXCEL_PATH.exists(),
}

for name, passed in checks.items():
    print(("PASS  " if passed else "FAIL  ") + name)

if not all(checks.values()):
    raise RuntimeError("At least one check failed. Read the list above.")

print("\nAll checks passed.", len(results), "papers in the evidence table.")

PASS  Records were downloaded
PASS  Evidence table has rows
PASS  All required columns are present
PASS  Every row has a PMID
PASS  No PMID appears twice
PASS  Every row has an evidence source
PASS  Every finding appears word for word in its abstract
PASS  CSV can be read back with the same number of rows
PASS  JSON can be read back with the same number of rows
PASS  Excel file exists

All checks passed. 50 papers in the evidence table.


## 15. Optional: a sample for manual validation

To find out how accurate the tool is, read some abstracts yourself and compare your answers with the tool's. This cell writes a sheet with a random sample of papers. The Study_Design, Population_Terms, Exposure_Terms and Outcome_Terms columns are left empty so that you can fill them in before you see what the tool said, and the tool's Main_Finding is shown in a separate column to judge afterwards. Filling in the sheet and scoring it happens in Manual_Validation.ipynb.

The sheet contains full abstracts, so keep it out of a public repository.

If you looked at certain papers while writing or adjusting the word lists, put their PMIDs in ALREADY_REVIEWED_PMIDS. A score measured on the papers used to build the rules is too optimistic, so those papers are left out of the sample. With MAX_RESULTS at 50 there may be fewer than 30 unseen papers left, and raising MAX_RESULTS gives you more to draw from.

In [ ]:
ALREADY_REVIEWED_PMIDS = []
SAMPLE_SIZE = 30
SAMPLE_SEED = 2025

VALIDATION_DIR = Path("validation")
VALIDATION_DIR.mkdir(exist_ok=True)
SHEET_PATH = VALIDATION_DIR / "annotation_sheet.csv"

seen = {str(pmid).strip() for pmid in ALREADY_REVIEWED_PMIDS}
candidates = evidence[evidence["Clean_Abstract"].str.strip().ne("") & ~evidence["PMID"].isin(seen)]
if candidates.empty:
    raise ValueError("No papers with an abstract are left to sample from.")

sample = candidates.sample(n=min(SAMPLE_SIZE, len(candidates)), random_state=SAMPLE_SEED)

sheet = pd.DataFrame({
    "PMID": sample["PMID"],
    "Title": sample["Title"],
    "Abstract": sample["Clean_Abstract"],
    "Study_Design": "",
    "Population_Terms": "",
    "Exposure_Terms": "",
    "Outcome_Terms": "",
    "Notes": "",
    "Tool_Main_Finding": sample["Main_Finding"],
    "Finding_Judgement": "",
}).reset_index(drop=True)
sheet.to_csv(SHEET_PATH, index=False, encoding="utf-8-sig")

print(f"Wrote {len(sheet)} papers to {SHEET_PATH}")
if len(sheet) < SAMPLE_SIZE:
    print(f"That is fewer than the {SAMPLE_SIZE} you asked for. Raise MAX_RESULTS to get more unseen papers.")

print("\nStudy_Design options (or write none):")
for label, _, _ in STUDY_DESIGNS:
    print("  ", label)
for field, terms in (("Population_Terms", POPULATION_TERMS), ("Exposure_Terms", EXPOSURE_TERMS),
                     ("Outcome_Terms", OUTCOME_TERMS)):
    print(f"\n{field} (separate with semicolons, or write none):")
    print("  ", "; ".join(terms))

try:
    from google.colab import files
    files.download(str(SHEET_PATH))
except ImportError:
    pass

Wrote 30 papers to validation/annotation_sheet.csv

Study_Design options (or write none):
   Meta-analysis
   Systematic review
   Randomised controlled trial
   Clinical trial (not randomised or not stated)
   Prospective cohort study
   Retrospective cohort study
   Cohort study
   Case-control study
   Cross-sectional study
   Longitudinal study
   Prospective study
   Retrospective study
   Observational study
   Diagnostic accuracy study
   Review (not systematic)

Population_Terms (separate with semicolons, or write none):
   women; men; patients; participants; adolescents; adults; children; subjects; controls; infertile; premenopausal; postmenopausal

Exposure_Terms (separate with semicolons, or write none):
   BMI; obesity; overweight; adiposity; waist circumference; weight loss; weight gain; physical activity; exercise; diet; metformin; insulin resistance; hyperinsulinemia; hyperandrogenism; testosterone; SHBG; smoking; intermittent fasting; ketogenic diet; low-carbohydrate di

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## What this tool cannot do

It reads abstracts, not full papers, so anything reported only in the full text is invisible to it.

A term being found is not the same as the study measuring it. The tool sees the word and nothing about the role the word played, which is why the columns are called Terms.

The main finding is chosen from wording patterns. The tool does not know whether a result was positive or negative, only that a sentence reads like a result, and it can choose the wrong sentence.

Study design comes from PubMed's labels and from design words in the text. A paper that does not state its design, or states it in unusual words, gets an empty Study_Design. In an abstract with no sections, a design that is only mentioned in the background can be picked up by mistake.

PubMed ranks the results and MAX_RESULTS keeps only the top of the list. A proper systematic review needs an exhaustive search, not one capped at a fixed number of papers.

The tool helps a person screen papers faster. It does not replace reading them.